# BookNooks dataset workspace

Run the cell below to load the newest collected CSV and display the first 50 book records as a dataframe. `books_50` keeps every field from the source file.

In [ ]:

import pandas as pd
from pathlib import Path
import re


# 1. FIND THE BOOKNOOKS PROJECT FOLDER


# In a Jupyter notebook, use the current working folder.
current_folder = Path.cwd()

# Find the project folder, even if the notebook
# is opened from a subfolder.
PROJECT_DIR = current_folder

for folder in [current_folder, *current_folder.parents]:
    if (folder / "data" / "processed").exists():
        PROJECT_DIR = folder
        break

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print(f"Project folder: {PROJECT_DIR}")



# FIND AND LOAD THE LATEST BOOK DATASET

# Find CSV files created by collect_books.py.
csv_files = list(PROCESSED_DIR.glob("books_sample_*.csv"))

# Also support a manually created book_sample.csv.
root_csv = PROJECT_DIR / "book_sample.csv"

if root_csv.exists():
    csv_file = root_csv
elif csv_files:
    csv_file = max(
        csv_files,
        key=lambda file: file.stat().st_mtime
    )
else:
    raise FileNotFoundError(
        "No book CSV found. Check data/processed "
        "or place book_sample.csv in the project folder."
    )

df = pd.read_csv(csv_file)

print(f"Dataset loaded: {csv_file.name}")
print(f"Number of books: {len(df)}")



# DEFINE MOOD KEYWORDS

mood_keywords = {
    "hopeful": [
        "hope", "hopeful", "optimistic", "triumph",
        "courage", "inspiring", "overcome", "dream"
    ],
    "dark": [
        "dark", "grim", "brutal", "bleak", "cruel",
        "death", "despair", "horror", "evil"
    ],
    "suspenseful": [
        "suspense", "danger", "threat", "mystery",
        "chase", "secret", "murder", "betrayal"
    ],
    "humorous": [
        "funny", "humor", "humour", "comedy",
        "comic", "hilarious", "witty"
    ],
    "romantic": [
        "romance", "romantic", "love", "passion",
        "beloved", "relationship"
    ],
    "reflective": [
        "reflect", "memory", "regret", "grief",
        "loss", "identity", "meaning", "loneliness"
    ],
    "adventurous": [
        "adventure", "journey", "quest", "explore",
        "battle", "warrior", "expedition", "travel"
    ],
    "mysterious": [
        "mysterious", "enigma", "puzzle", "unknown",
        "uncover", "investigate"
    ]
}


# ==========================================
# 4. ASSIGN MOOD TAGS FROM DESCRIPTIONS
# ==========================================

def tag_mood(description):
    """Find mood-related keywords in a book description."""

    if pd.isna(description) or not str(description).strip():
        return "unknown"

    text = str(description).lower()
    found_tags = []

    for mood, keywords in mood_keywords.items():
        for keyword in keywords:

            # Match whole words or word beginnings,
            # avoiding matches inside unrelated words.
            pattern = r"\b" + re.escape(keyword)

            if re.search(pattern, text):
                found_tags.append(mood)
                break

    if found_tags:
        return "; ".join(found_tags)

    return "undetermined"


# Create the new column.
if "description" in df.columns:
    df["mood_tags"] = df["description"].apply(tag_mood)
else:
    df["mood_tags"] = "unknown"
    print("Warning: No description column was found.")



# UNNECESSARY DISPLAY COLUMNS


df = df.drop(
    columns=["source_url", "rights_notes"],
    errors="ignore"
)



# ORGANIZE THE TABLE COLUMNS


preferred_columns = [
    "title",
    "author",
    "publication_year",
    "description",
    "mood_tags",
    "genres",
    "description_word_count",
    "book_id",
    "details_retrieved"
]

# Put the important columns first.
existing_columns = [
    column for column in preferred_columns
    if column in df.columns
]

# Keep any additional fields that are present.
other_columns = [
    column for column in df.columns
    if column not in existing_columns
]

df = df[existing_columns + other_columns]



# DISPLAY THE BOOKS AS A TABLE


print("\nBOOKNOOK BOOK PREVIEW")
print(f"Total books: {len(df)}")

# Display the actual interactive-style notebook table.
# Long descriptions are shortened only for display.
display(
    df.style.set_properties(
        subset=["description"],
        **{
            "max-width": "450px",
            "white-space": "normal",
            "text-align": "left"
        }
    ) if "description" in df.columns else df
)



# SAVE A SEPARATE CLEANED DATASET


output_file = PROJECT_DIR / "book_sample_with_moods.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nPreview saved to: {output_file}")
print("Original dataset has not been modified.")
print(
    "Note: Mood tags are provisional keyword matches, "
    "not validated sentiment predictions."
)


Project folder: /Users/motunrayo.a/Desktop/BookNooks
Dataset loaded: books_sample_20261010_044503.csv
Number of books: 100

BOOKNOOK BOOK PREVIEW
Total books: 100


,title,author,publication_year,description,mood_tags,genres,description_word_count,book_id,details_retrieved
0,The Blade Itself,Joe Abercrombie,2001,"Logen Ninefingers, infamous barbarian, has finally run out of luck. Caught in one feud too many, he’s on the verge of becoming a dead barbarian – leaving nothing behind him but bad songs, dead friends, and a lot of happy enemies. Nobleman Captain Jezal dan Luthar, dashing officer, and paragon of selfishness, has nothing more dangerous in mind than fleecing his friends at cards and dreaming of glory in the fencing circle. But war is brewing, and on the battlefields of the frozen North they fight by altogether bloodier rules. Inquisitor Glokta, cripple turned torturer, would like nothing better than to see Jezal come home in a box. But then Glokta hates everyone: cutting treason out of the Union one confession at a time leaves little room for friendship. His latest trail of corpses may lead him right to the rotten heart of government, if he can stay alive long enough to follow it. Enter the wizard, Bayaz. A bald old man with a terrible temper and a pathetic assistant, he could be the First of the Magi, he could be a spectacular fraud, but whatever he is, he's about to make the lives of Logen, Jezal, and Glotka a whole lot more difficult. Murderous conspiracies rise to the surface, old scores are ready to be settled, and the line between hero and villain is sharp enough to draw blood.",hopeful; suspenseful; adventurous,"Fiction, fantasy, general; Fantasy fiction; Fantasy; Fiction",230,/works/OL8400950W,True
1,The Hero of Ages,Brandon Sanderson,2008,"This book is not only the third book in a trilogy, but it’s Act Three of the the three-act structure for the Mistborn Trilogy—it’s the part of the story where the heroes have discovered that what they thought was the problem all along was not the true danger, and now they’re fighting for not only their own survival but that of the world they live in. The mists are killing people and staying out much longer than they should. The Ashmounts are spewing more and more choking ash into the sky, burying the crops that everyone needs to eat to live. And Ruin, the creature Vin was tricked into freeing from its prison of a millennium, is loose to wreak havoc upon the land. Life under the Lord Ruler is starting to look like paradise in comparison. While the first book in the trilogy turned the standard fantasy story on its head, this volume (perhaps inevitably?) returns in a way to the tropes the first volume was a reaction against. Yet in this case the enemy is not a human or humanlike Dark Lord, but something more like a force of nature—entropy itself given a will and a guiding personality in the form of Ruin. Ultimately, the book is about how the characters we have grown to love from the previous volumes—Vin, Elend, Sazed, TenSoon, Spook, Marsh, and others—find the courage and faith to fight on in the face of overwhelming odds, just as Kelsier taught them when he plotted the downfall of the Final Empire.",hopeful; dark; suspenseful; romantic,"Fiction; Fantasy; Epic Fantasy; Fantasy fiction; Fiction, fantasy, general; Children's fiction; Imaginary places; Magic",256,/works/OL5738154W,True
2,The Lies of Locke Lamora,Scott Lynch,2001,Best book ever,undetermined,"Fiction, fantasy, general; Fantasy; Fiction; American Science Fiction And Fantasy; Fiction - Fantasy; Fantasy - Epic; Fiction / Fantasy / Epic; Juvenile fiction; Gangsters; Swindlers and swindling; Orphans; Adventure stories; Fantasy fiction; Robbers and outlaws; Fiction, fantasy, epic",3,/works/OL8369445W,True
3,The Assassin’s Blade,Sarah J. Maas,2014,"Celaena Sardothien owes her reputation to Arobynn Hamel. He gave her a home at the Assassins' Guild and taught her the skills she needed to survive. Arobynn's enemies stretch far and wide - from Adarlan's rooftops and its filthy dens, to remote islands and hostile deserts. Celaena is duty-bound to hunt them down. But behind her assignments l


Preview saved to: /Users/motunrayo.a/Desktop/BookNooks/book_sample_with_moods.csv
Original dataset has not been modified.
Note: Mood tags are provisional keyword matches, not validated sentiment predictions.


# Step-by-step recommender walkthrough

The first section above is the original data exploration. Run this walkthrough from top to bottom to inspect the collection, quality counts, mood review process, and recommendation strategies. Reusable implementations remain in the `src` modules; these cells call them so you can inspect each result. The optional collection cell makes live Open Library requests and is off by default.


## 1. Find the project and load the catalog

This works whether Jupyter starts in the repository root or in the notebooks folder. The mood-enriched catalog contains source URLs and rights notes as well as recommendation fields.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

current = Path.cwd().resolve()
PROJECT_DIR = next(
    (folder for folder in (current, *current.parents)
     if (folder / "src" / "recommendation").is_dir()
     and (folder / "data" / "processed").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("Start Jupyter from this repository or its notebooks folder")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

CATALOG_PATH = PROJECT_DIR / "data" / "processed" / "books_sample_5000_with_moods.csv"
books_df = pd.read_csv(CATALOG_PATH).fillna("")
print(f"Project: {PROJECT_DIR}")
print(f"Catalog: {CATALOG_PATH.relative_to(PROJECT_DIR)}")
print(f"Rows: {len(books_df)} | unique book IDs: {books_df['book_id'].nunique()}")
display(books_df[["title", "author", "genres", "mood_tags", "source_url", "rights_notes"]].head())


## 2. Recompute the saved-record and quality counts

These counts are calculated from the CSV itself. A description is counted as usable when it has at least 50 words. Mood derivability is not the same as validated mood labels.


In [ ]:
from src.evaluation.data_quality import summarize

quality = summarize(CATALOG_PATH, requested=100)
display(pd.Series(quality, name="value").to_frame())


## 3. Reproduce collection (optional)

Set RUN_COLLECTION to True only when you want to query Open Library. The collector saves timestamped raw responses, processed records, and a quality report. It pauses between work-detail requests.


In [ ]:
RUN_COLLECTION = False  # Change to True to fetch a new sample

if RUN_COLLECTION:
    from datetime import datetime, timezone
    from src.data.openlibrary import fetch_search_results, collect_records, write_collection

    query, target = "subject:fantasy", 100
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    raw_dir = PROJECT_DIR / "data" / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    raw_search = raw_dir / f"search_{timestamp}.json"
    raw_details = raw_dir / f"work_details_{timestamp}"
    search_data = fetch_search_results(query=query, limit=target)
    collected = collect_records(search_data, raw_details_dir=raw_details)
    saved_csv, saved_report, collection_summary = write_collection(
        collected, search_data, raw_search, raw_details,
        PROJECT_DIR / "data" / "processed", target, query
    )
    display(pd.Series(collection_summary, name="value").to_frame())
else:
    print("Skipped live collection. Set RUN_COLLECTION = True when ready.")


## 4. Review mood tags with human judgments

Mood labels were created heuristically and are not yet validated. The next cell creates a reproducible 25-book annotation sheet with blank human labels. Fill in human_mood_tags with semicolon-separated labels, then evaluate the completed sheet. Blank labels do not count as reviews.


In [ ]:
from src.evaluation.mood_validation import make_template, evaluate

MOOD_REVIEW_PATH = PROJECT_DIR / "results" / "experiments" / "mood_review_template.csv"
if not MOOD_REVIEW_PATH.exists():
    make_template(CATALOG_PATH, MOOD_REVIEW_PATH, sample_size=25, seed=42)
else:
    print("Existing review sheet found; leaving it unchanged so annotations are preserved.")
print(f"Review sheet: {MOOD_REVIEW_PATH.relative_to(PROJECT_DIR)}")
print("Allowed tags: hopeful, dark, suspenseful, adventurous, romantic, reflective, mysterious, humorous, undetermined")
print(evaluate(MOOD_REVIEW_PATH))


## 5. Inspect the four reader profiles

Profiles provide a starting point. You can add or replace genres and moods in the interactive preference cell below.


In [ ]:
from src.recommendation.engine import READER_PROFILES, recommend

display(pd.DataFrame.from_dict(READER_PROFILES, orient="index").rename_axis("reader_profile"))


## 6. Choose a profile and personalize it

Run this cell to select a profile (1–4) or custom preferences. In a notebook, answer each prompt in the input box that appears.


In [ ]:
profile_names = list(READER_PROFILES)
for number, name in enumerate(profile_names, start=1):
    print(f"{number}. {name.title()} — {READER_PROFILES[name]['description']}")
print("5. Custom preferences")
profile_choice = input("Choose a profile number: ").strip()

if profile_choice == "5":
    selected_profile = "custom"
    default_genres, default_moods = "", ""
elif profile_choice in {"1", "2", "3", "4"}:
    selected_profile = profile_names[int(profile_choice) - 1]
    default_genres = READER_PROFILES[selected_profile]["genres"]
    default_moods = READER_PROFILES[selected_profile]["moods"]
else:
    raise ValueError("Choose 1, 2, 3, 4, or 5")

genre_preferences = input(f"Genres [{default_genres}]: ").strip() or default_genres
mood_preferences = input(f"Moods [{default_moods}]: ").strip() or default_moods
keyword_preferences = input("Other themes or keywords (optional): ").strip()
print(f"Using profile: {selected_profile}")


## 7. Run the genre-only baseline

The baseline ranks books by overlap between requested genre words and the catalog's subject terms. It does not use mood tags or TF-IDF.


In [ ]:
genre_results = recommend(
    books_df.to_dict(orient="records"),
    genres=genre_preferences,
    moods=mood_preferences,
    query=keyword_preferences,
    strategy="genre",
    limit=5,
)
genre_top5 = pd.DataFrame(genre_results)
display(genre_top5[["title", "author", "matched_genres", "recommendation_score", "source_url", "rights_notes"]])


## 8. Compare experimental ranking strategies

This displays the current top five from each method for the same preferences. It is a mechanics comparison, not evidence that either experimental method is better. For that claim, collect human relevance judgments for the same queries and compare the methods on those judgments.


In [ ]:
tfidf_results = recommend(
    books_df.to_dict(orient="records"), genres=genre_preferences,
    moods=mood_preferences, query=keyword_preferences,
    strategy="tfidf", limit=5,
)
mood_results = recommend(
    books_df.to_dict(orient="records"), genres=genre_preferences,
    moods=mood_preferences, query=keyword_preferences,
    strategy="mood", limit=5,
)

comparison = pd.concat([
    pd.DataFrame({"strategy": "genre baseline", "rank": range(1, 6), "title": [b["title"] for b in genre_results]}),
    pd.DataFrame({"strategy": "TF-IDF experiment", "rank": range(1, 6), "title": [b["title"] for b in tfidf_results]}),
    pd.DataFrame({"strategy": "mood experiment", "rank": range(1, 6), "title": [b["title"] for b in mood_results]}),
], ignore_index=True)
display(comparison.pivot(index="rank", columns="strategy", values="title"))


## 9. What to record

When human mood review or reader relevance judgments are complete, record the catalog version, queries, strategy settings, and metrics in the research log. Until then, report saved-record counts and demonstrate rankings, but do not claim that mood weighting or TF-IDF improves recommendations.


## 10. Inspect the reusable implementations

The notebook calls these same functions used by the terminal app and collection script. Display their source when you want to follow the implementation line by line.


In [ ]:
import inspect
from src.data import openlibrary
from src.recommendation import engine

print("COLLECTION: collect_records")
print(inspect.getsource(openlibrary.collect_records))
print("RECOMMENDATIONS: recommend")
print(inspect.getsource(engine.recommend))


## 11. Try the chat-style UI

For a visual chat interface, open a terminal at the repository root and run `streamlit run src/recommendation/chat_app.py`. Select a profile and ranking method, then describe what you want to read. The UI uses the same recommendation engine and catalog as the cells above.
